In [1]:
import torch, librosa, torch.nn as nn
from mpmath import diffun
from torch.utils.data import  DataLoader
import numpy as np
import math, glob, json

from model import CustomDataset, SoundSequenceAnalyzer, mdn_loss


In [5]:
def train(model, train_loader, test_loader ,device="cpu", epochs=200, lr=0.001, debug=False):
    print("Using device: ", device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    #
    is_object_lf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.0]).to(device))#missed true positive get 2x penalty
    obj_type_lf = nn.CrossEntropyLoss()
    attr_lf = nn.SmoothL1Loss()
    curve_lf = nn.CrossEntropyLoss()
    anchor_count_lf = nn.CrossEntropyLoss()
    anchor_coord_lf = nn.MSELoss(reduction='none')


    train_losses, val_losses = [], []
    best_loss = float('inf')
    epochs_wo_improvement = 0
    model = model.to(device)
    for epoch in range(epochs):
        model.train()
        if epochs_wo_improvement>9:
            print("Early stopping at epoch", epoch)
            break

        epoch_loss = 0.0
        for X, y, difficulty in train_loader:
            X, y, difficulty = X.to(device), y.to(device), difficulty.to(device)
            loss_coords = 0.0
            loss_obj_type = 0.0
            loss_attr = 0.0
            loss_curve = 0.0
            loss_anchor_count = 0.0
            loss_anchor_coords = 0.0
            is_object = y[:, :, 0].unsqueeze(-1)
            coords = y[:, :, 1:3]
            obj_type = y[:, :, 3:6]   # 3 classes: circle, slider, spinner
            obj_attr = y[:, :, 6:9]
            curve_targets = y[:, :, 9].long()
            is_obj_p, coords_p, type_p, attr_p, curve_p, anchor_count_p, anchor_coords_p = model(X, difficulty)
            loss_obj_all = is_object_lf(is_obj_p, is_object )

            mask = (is_object == 1).squeeze(-1)
            if mask.any():
                #loss_coords = coords_lf(coords_pred[mask], coords[mask])
                pi_logits, mu, log_sigma = coords_p
                loss_coords = mdn_loss(pi_logits[mask], mu[mask], log_sigma[mask], coords[mask])

                valid_type_preds = type_p[mask]
                valid_type_targets = obj_type[mask]
                loss_obj_type = obj_type_lf(valid_type_preds, valid_type_targets.argmax(dim=-1))
                loss_attr = attr_lf(attr_p[mask], obj_attr[mask])

                slider_mask = (obj_type[mask][:, 1] == 1.0)

                if slider_mask.any():
                    loss_curve = curve_lf(curve_p[mask][slider_mask], curve_targets[mask][slider_mask])
                    #Anchor Count Loss
                    target_anchor_counts = y[mask][slider_mask, 10].long()
                    loss_anchor_count = anchor_count_lf(anchor_count_p[mask][slider_mask], target_anchor_counts)
                    #Masked Coordinate Loss
                    pred_anchors = anchor_coords_p[mask][slider_mask]
                    targ_anchors = y[mask][slider_mask, 11:17]

                    # Create a boolean mask of shape [N, 6] denoting which coordinates are valid
                    valid_coords_mask = torch.zeros_like(pred_anchors, dtype=torch.bool)
                    for i, count in enumerate(target_anchor_counts):
                        if count > 0:
                            valid_coords_mask[i, :count*2] = True

                    raw_coord_loss = anchor_coord_lf(pred_anchors, targ_anchors)
                    # Average the loss only over the valid anchor coordinates
                    loss_anchor_coords = raw_coord_loss[valid_coords_mask].mean() if valid_coords_mask.any() else 0.0
                else:
                    loss_curve = 0.0
                    loss_anchor_count = 0.0
                    loss_anchor_coords = 0.0
            else:
                loss_obj_type = 0.0
                loss_coords = 0.0
                loss_attr = 0.0
                loss_curve = 0.0

            masked_loss_sum = loss_coords + loss_obj_type + loss_attr + loss_curve + loss_anchor_coords + loss_anchor_count
            total_loss = loss_obj_all + (1 * masked_loss_sum)
            if debug: print(f"Loss obj: {loss_obj_all}, loss_coords: {loss_coords}, loss_obj_type: {loss_obj_type}")
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            epoch_loss += total_loss.item() * len(X)

        avg_train_loss = epoch_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)


        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X, y, difficulty in test_loader:
                X, y, difficulty = X.to(device), y.to(device), difficulty.to(device)
                is_object = y[:, :, 0].unsqueeze(-1)
                coords = y[:, :, 1:3]
                obj_type = y[:, :, 3:6]   # 3 classes: circle, slider, spinner
                obj_attr = y[:, :, 6:9]
                curve_targets = y[:, :, 9].long()

                is_obj_p, coords_p, type_p, attr_p, curve_p, anchor_count_p, anchor_coords_p = model(X, difficulty)
                loss_obj_all = is_object_lf(is_obj_p, is_object )

                mask = (is_object == 1).squeeze(-1)
                if mask.any():
                    pi_logits, mu, log_sigma = coords_p
                    loss_coords = mdn_loss(pi_logits[mask], mu[mask], log_sigma[mask], coords[mask])

                    valid_type_preds = type_p[mask]
                    valid_type_targets = obj_type[mask]
                    loss_obj_type = obj_type_lf(valid_type_preds, valid_type_targets.argmax(dim=-1))
                    loss_attr = attr_lf(attr_p[mask], obj_attr[mask])

                    slider_mask = (obj_type[mask][:, 1] == 1.0)

                    if slider_mask.any():
                        loss_curve = curve_lf(curve_p[mask][slider_mask], curve_targets[mask][slider_mask])
                        #Anchor Count Loss
                        target_anchor_counts = y[mask][slider_mask, 10].long()
                        loss_anchor_count = anchor_count_lf(anchor_count_p[mask][slider_mask], target_anchor_counts)
                        #Masked Coordinate Loss
                        pred_anchors = anchor_coords_p[mask][slider_mask]
                        targ_anchors = y[mask][slider_mask, 11:17]

                        # Create a boolean mask of shape [N, 6] denoting which coordinates are valid
                        valid_coords_mask = torch.zeros_like(pred_anchors, dtype=torch.bool)
                        for i, count in enumerate(target_anchor_counts):
                            if count > 0:
                                valid_coords_mask[i, :count*2] = True

                        raw_coord_loss = anchor_coord_lf(pred_anchors, targ_anchors)
                        # Average the loss only over the valid anchor coordinates
                        loss_anchor_coords = raw_coord_loss[valid_coords_mask].mean() if valid_coords_mask.any() else 0.0
                    else:
                        loss_curve = 0.0
                        loss_anchor_count = 0.0
                        loss_anchor_coords = 0.0
                else:
                    loss_obj_type = 0.0
                    loss_coords = 0.0
                    loss_attr = 0.0
                    loss_curve = 0.0
                masked_loss_sum = loss_coords + loss_obj_type + loss_attr + loss_curve + loss_anchor_coords + loss_anchor_count
                total_loss = loss_obj_all + (1 * masked_loss_sum)
                val_loss += total_loss.item() * len(X)

        avg_val_loss = val_loss / len(test_loader.dataset)
        val_losses.append(avg_val_loss)
        print(f"Epoch {epoch:>3} | train loss: {avg_train_loss:.4f} | val loss: {avg_val_loss:.4f}")
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            epochs_wo_improvement=0
            #torch.save(model.state_dict(), "weights.pth")
        else:
            #epochs_wo_improvement += 1
            pass

    else:
        print(f"Epoch {epoch:>3} | train loss: {avg_train_loss:.4f}")

    torch.cuda.empty_cache()

    return train_losses, val_losses


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#all_samples = glob.glob(r"./processed/**/*.json", recursive=True)
#cust_dataset = CustomDataset(all_samples, hop_length=1024)
train_dataset, test_dataset = CustomDataset.split(
    n_samples=80,
    train_ratio=0.8,
    seed=42,
    window_size=1024,
    debug=False
)


train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=8, shuffle=False)

#model = SoundSequenceAnalyzer(time_window_size=1024, freq_vec_len=128, num_heads=16)
model = SoundSequenceAnalyzer(time_window_size=1024, freq_vec_len=256, num_heads=32)
state_dict = torch.load('w_4+7+9.pth', weights_only=True, map_location=torch.device('cpu'))
model.load_state_dict(state_dict)
train(model, train_loader=train_dataloader, test_loader=test_dataloader, device=device, epochs=100, lr=0.001, debug=False)


In [ ]:
sample_track, _ = CustomDataset.split(
    n_samples=1,
    train_ratio=1,
    seed=42,
    window_size=1024,
    debug=True,
    inference=False
)
sample_loader = DataLoader(sample_track, batch_size=2, shuffle=False)
for X, y, difficulty in sample_loader:
    break

#model = SoundSequenceAnalyzer(time_window_size=1024, freq_vec_len=100, num_heads=10)
#model(X, difficulty)

In [ ]:
torch.cuda.empty_cache()


In [ ]:
model = SoundSequenceAnalyzer(time_window_size=4096, freq_vec_len=100, num_heads=10)
state_dict = torch.load('weights.pth', weights_only=True, map_location=torch.device('cpu'))
model.load_state_dict(state_dict)
model.eval()

In [ ]:

model.eval()
model.to(device)
    
tp = 0
fp = 0
fn = 0    
with torch.no_grad():
    for X, y, difficulty in test_dataloader:
        X = X.to(device)
        difficulty = difficulty.to(device)
        
        outputs = model(X, difficulty)
        is_object_pred = outputs[0].to('cpu') # [B, Window_Size, 1]
        
        pred_logits = is_object_pred.squeeze(-1)
        actual_targets = y[:, :, 0] # Ground truth element activity flag
        
        predicted_positive = (pred_logits > threshold)
        actual_positive = (actual_targets == 1.0)
      
        tp += (predicted_positive & actual_positive).sum().item()
        fp += (predicted_positive & ~actual_positive).sum().item()
        fn += (~predicted_positive & actual_positive).sum().item()

# Calculate metrics with zero-division protection fallbacks
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
print(f"Precision: {precision}, Recall: {recall}, f1: {f1_score}")
print(f"TP: {tp} FP:{fp} FN: {fn}")

In [ ]:
all_samples = glob.glob(r"./processed/**/*.json", recursive=True)
for idx in range(len(all_samples)):
    if '2138603' in all_samples[idx]:
        print(idx)

In [10]:
def process_long_sequence(model, X, difficulty, window_size, device="cpu"):
    """
    Slices an audio track sequentially into window_size segments,
    evaluates them via the model, and joins the final predictions.
    """
    model.eval()
    model.to(device)

    # Ensure standard batch dimensions exist [B, Features, Time]
    if X.dim() == 2:
        X = X.unsqueeze(0)
    if difficulty.dim() == 2:
        difficulty = difficulty.unsqueeze(0)

    total_frames = X.shape[-1]
    num_chunks = total_frames // window_size

    # Storage arrays for collecting outputs across loops
    all_is_object = []
    all_pi_logits = []
    all_mu = []
    all_log_sigma = []
    all_obj_types = []
    all_obj_attrs = []
    all_curve_types = []
    all_anchor_count = []
    all_anchor_coords = []
    with torch.no_grad():
        for i in range(num_chunks):
            start = i * window_size
            end = start + window_size

            # Slice step
            X_chunk = X[..., start:end].to(device)
            diff_chunk = difficulty[:, start:end, :].to(device)

            # Evaluate forward pass
            outputs = model(X_chunk, diff_chunk)

            # Accommodates both 3-output and 4-output model versions cleanly

            is_obj_pred, mdn_out, obj_type_pred, obj_attr_pred, curve_type, anchor_count, anchor_coords = outputs
            all_obj_attrs.append(obj_attr_pred.cpu())


            pi_logits, mu, log_sigma = mdn_out

            # Keep on CPU to preserve VRAM during loop steps
            all_is_object.append(is_obj_pred.cpu())
            all_pi_logits.append(pi_logits.cpu())
            all_mu.append(mu.cpu())
            all_log_sigma.append(log_sigma.cpu())
            all_obj_types.append(obj_type_pred.cpu())
            all_curve_types.append(curve_type.cpu())
            all_anchor_coords.append(anchor_coords.cpu())
            all_anchor_count.append(anchor_count.cpu())

    # Concatenate chunk blocks back into standard continuous timelines (dim 1)
    full_is_object = torch.cat(all_is_object, dim=1)
    full_pi_logits = torch.cat(all_pi_logits, dim=1)
    full_mu = torch.cat(all_mu, dim=1)
    full_log_sigma = torch.cat(all_log_sigma, dim=1)
    full_obj_types = torch.cat(all_obj_types, dim=1)
    full_curve_types = torch.cat(all_curve_types, dim=1)
    full_anchor_count = torch.cat(all_anchor_count, dim=1)
    full_anchor_coords = torch.cat(all_anchor_coords, dim=1)
    mdn_recombined = (full_pi_logits, full_mu, full_log_sigma)

    if all_obj_attrs:
        full_obj_attrs = torch.cat(all_obj_attrs, dim=1)
        return (full_is_object, mdn_recombined, full_obj_types, full_obj_attrs, full_curve_types, full_anchor_count, full_anchor_coords)

    return full_is_object, mdn_recombined, full_obj_types

sample_track, _ = CustomDataset.split(
    n_samples=1,
    train_ratio=1,
    seed=42,
    window_size=8096,
    debug=True,
    inference=True
)
sample_loader = DataLoader(sample_track, batch_size=1, shuffle=False)
for X, y, difficulty in sample_loader:
    break
model_out = process_long_sequence(model, X, difficulty, window_size=1024)
is_object_p = model_out[0]
pi_logits, mu, log_sigma = model_out[1]
obj_type_p = model_out[2]
obj_attr_p = model_out[3]
curve_type_p = model_out[4]
anchor_count_p = model_out[5]
anchor_coords_p = model_out[6]

Using debug config
2382142/5150129


In [15]:


def mdn_sample(pi_logits, mu, log_sigma, temperature=1.0):
    """
    Stochastically samples coordinates from the MDN parameters to inject variety.
    temperature: >1.0 increases randomness, <1.0 makes it closer to standard flow.
    """
    # 1. Sample a mixture component index based on pi probabilities
    pi_dist = torch.distributions.Categorical(logits=pi_logits / temperature)
    component_idx = pi_dist.sample() # Shape: [N]

    # 2. Gather the corresponding mu and log_sigma parameters
    N, K, D = mu.shape  # Batch/Masked elements, Components, Dimensions (2)
    idx = component_idx.unsqueeze(-1).unsqueeze(-1).expand(N, 1, D)

    chosen_mu = mu.gather(-2, idx).squeeze(-2)
    chosen_log_sigma = log_sigma.gather(-2, idx).squeeze(-2)
    chosen_sigma = torch.exp(chosen_log_sigma)

    # 3. Sample from the selected normal distribution
    epsilon = torch.randn_like(chosen_mu)
    sampled_coords = chosen_mu + (epsilon * chosen_sigma * temperature)

    return sampled_coords
sample_rate = 22050
hop_length = 1024

mask = (is_object_p > 0).squeeze(-1)

if mask.any():
    coords_out = mdn_sample(pi_logits[mask], mu[mask], log_sigma[mask], temperature=0.85)

    # Extract predicted structural types (0=Circle, 1=Slider, 2=Spinner)
    pred_classes = torch.argmax(obj_type_p[mask], dim=-1)

    # Extract geometry mappings across frames
    masked_attributes = obj_attr_p[mask]
    pred_curves = torch.argmax(curve_type_p[mask], dim=-1)
    pred_anchor_counts = torch.argmax(anchor_count_p[mask], dim=-1)
    masked_anchors = anchor_coords_p[mask]

    # Calculate global timestamps in milliseconds from timeline frames
    mask_idxs = torch.where(mask == True)
    ms = (mask_idxs[1].numpy() * (1000 * hop_length) / sample_rate).astype(int)


In [ ]:
# --- Complete Upgraded Inference Loop ---

model = model.to('cpu')

if mask.any():
    # FIXED: Apply Stochastic Sampling to completely eliminate Center Position Collapse
    coords_out = mdn_sample(pi_logits[mask], mu[mask], log_sigma[mask], temperature=0.85)

    # Extract predicted structural types (0=Circle, 1=Slider, 2=Spinner)
    pred_classes = torch.argmax(obj_type_p[mask], dim=-1)

    # Extract geometry mappings across frames
    masked_attributes = obj_attr_p[mask]
    pred_curves = torch.argmax(curve_type_p[mask], dim=-1)
    pred_anchor_counts = torch.argmax(anchor_count_p[mask], dim=-1)
    masked_anchors = anchor_coords_p[mask]

    # Calculate global timestamps in milliseconds from timeline frames
    mask_idxs = torch.where(mask == True)
    sample_rate = 22050
    hop_length = 1024
    ms = (mask_idxs[1].numpy() * (1000 * hop_length) / sample_rate).astype(int)

    # Step 2: Build the structural object arrays
    generated_hit_objects = []

    for idx, time_ms in enumerate(ms):
        # Denormalize stochastic starting points
        x = max(0, min(512, int(512 * coords_out[idx, 0].item())))
        y = max(0, min(384, int(384 * coords_out[idx, 1].item())))

        model_class = pred_classes[idx].item()

        obj = {
            "x": x,
            "y": y,
            "time": int(time_ms)
        }

        # Map class integers back to official osu! bit flags
        if model_class == 0:
            obj["type"] = 1 # Circle Flag

        elif model_class == 1:
            obj["type"] = 2 # Slider Flag

            # Extract basic length parameters
            obj["length"] = max(10.0, float(masked_attributes[idx, 2].item() * 1000.0))
            obj["curve_class_idx"] = int(pred_curves[idx].item())

            # Extract anchor counts and their respective coordinates
            num_anchors = int(pred_anchor_counts[idx].item())
            obj["anchor_coords"] = []

            for j in range(num_anchors):
                ax = max(0, min(512, int(512 * masked_anchors[idx, j*2].item())))
                ay = max(0, min(384, int(384 * masked_anchors[idx, (j*2)+1].item())))
                obj["anchor_coords"].append((ax, ay))

            # Keep a fallback traditional endpoint just in case num_anchors calculates to 0
            obj["end_x"] = max(0, min(512, int(512 * masked_attributes[idx, 0].item())))
            obj["end_y"] = max(0, min(384, int(384 * masked_attributes[idx, 1].item())))

        elif model_class == 2:
            obj["type"] = 8 # Spinner Flag
            duration_ms = max(100, int(masked_attributes[idx, 2].item() * 1000.0))
            obj["endTime"] = int(time_ms + duration_ms)

        generated_hit_objects.append(obj)

    # Inject objects back into the target format dict
    all_samples = glob.glob(r"./processed/**/*.json", recursive=True)
    with open(all_samples[1], 'r') as file:
        data = json.load(file)

    data["hit_objects"] = generated_hit_objects

    # Save file
    audio_path = "./sample_map/1022180/audio.mp3"
    json_to_osu(data, "tmp.osu", audio_path)
else:
    print("Zero objects matched structural selection criteria thresholds.")

In [ ]:
import json
import os

def json_to_osu(parsed_data, output_path, audio_path):
    """
    Converts a parsed dictionary/JSON back into a valid .osu file format.

    :param parsed_data: Dict containing metadata, general, difficulty, and hit_objects.
    :param output_path: Path where the output .osu file will be saved.
    """
    lines = []

    # 1. Write the mandatory file format header (defaulting to v14)
    lines.append("osu file format v14")
    lines.append("")  # Blank line after header

    # 2. Reconstruct Key-Value Sections
    # Order matters slightly for readability, though the game is flexible
    kv_sections = ["general", "metadata", "difficulty"]
    for section_key in kv_sections:
        if section_key in parsed_data and parsed_data[section_key]:
            # Capitalize section name to match standard .osu style (e.g., [General])
            lines.append(f"[{section_key.capitalize()}]")
            for key, value in parsed_data[section_key].items():
                lines.append(f"{key}: {value}")
            lines.append("")  # Blank line after section

    # Generate Timing Point via Audio
    y_audio, sr = librosa.load(audio_path)
    tempo, _ = librosa.beat.beat_track(y=y_audio, sr=sr)
    bpm = float(tempo[0]) if isinstance(tempo, np.ndarray) else float(tempo)
    beat_length = 60000.0 / bpm if bpm > 0 else 500.0

    lines.append("[TimingPoints]")
    # Format: time, beatLength, meter, sampleSet, sampleIndex, volume, uninherited, effects
    lines.append(f"0,{beat_length},4,2,1,60,1,0")
    lines.append("")

    # Reconstruct HitObjects
    if "hit_objects" in parsed_data and parsed_data["hit_objects"]:
        lines.append("[HitObjects]")
        for obj in parsed_data["hit_objects"]:
            x = max(0, min(512, int(obj.get("x", 256))))
            y = max(0, min(384, int(obj.get("y", 192))))
            time = int(obj.get("time", 0))
            obj_type = int(obj.get("type", 1)) # Extracted from the model's argmax(obj_type)

            # Base parameters
            parts = [str(x), str(y), str(time), str(obj_type), "0"]

            inv_curve_map = {0: 'L', 1: 'B', 2: 'P', 3: 'C'}

            if obj_type == 2: # Slider (Assuming argmax gave us 2 for bit flag 2)
                length = max(10.0, float(obj.get("length", 100.0)))
                curve_letter = inv_curve_map.get(int(obj.get("curve_class_idx", 0)), 'L')

                # Fetch structural coordinates array
                anchors = obj.get("anchor_coords", [])

                if len(anchors) > 0:
                    # Construct multi-node anchor string format (e.g., B|200:100|250:300)
                    anchor_strings = [f"{ax}:{ay}" for ax, ay in anchors]
                    curve_syntax = f"{curve_letter}|" + "|".join(anchor_strings)
                else:
                    # Traditional fallback endpoint if zero anchors were predicted
                    end_x = max(0, min(512, int(obj.get("end_x", x))))
                    end_y = max(0, min(384, int(obj.get("end_y", y))))
                    curve_syntax = f"{curve_letter}|{end_x}:{end_y}"

                parts.append(curve_syntax)
                parts.append("1") # Default standard repeat slider loop count
                parts.append(str(length))

            elif obj_type == 8: # Spinner
                end_time = time + max(100, int(obj.get("attr_val", 1000)))
                parts.append(str(end_time))

            lines.append(",".join(parts))
        lines.append("")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"Successfully generated map with {len(generated_hit_objects)} structural elements!")


In [ ]:
tmp = 1

In [ ]:
print(tmp)